In [ ]:
# 导入 fastai 常用库；设置 numpy 打印宽度
from fastai.imports import *
np.set_printoptions(linewidth=130)

In [ ]:
# 下载 Titanic 数据；读训练/测试集；算每列众数用于填缺失
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')

if iskaggle: path = Path('../input/titanic')
else:
    import zipfile,kaggle
    path = Path('titanic')
    kaggle.api.competition_download_cli(str(path))
    zipfile.ZipFile(f'{path}.zip').extractall(path)

df = pd.read_csv(path/'train.csv')
tst_df = pd.read_csv(path/'test.csv')
modes = df.mode().iloc[0]

In [ ]:
# 数据预处理：填缺失、Fare 取对数、Sex/Embarked 转成分类类型
def proc_data(df):
    df['Fare'] = df.Fare.fillna(0)
    df.fillna(modes, inplace=True)
    df['LogFare'] = np.log1p(df['Fare'])
    df['Embarked'] = pd.Categorical(df.Embarked)
    df['Sex'] = pd.Categorical(df.Sex)

proc_data(df)
proc_data(tst_df)

In [ ]:
# 定义类别列 cats、连续列 conts、因变量 dep
cats=["Sex","Embarked"]
conts=['Age', 'SibSp', 'Parch', 'LogFare',"Pclass"]
dep="Survived"

In [ ]:
# 看 Sex 列原始值
df.Sex.head()

In [ ]:
# 看 Sex 转成分类编码后的整数值
df.Sex.cat.codes.head()

In [ ]:
# 用 seaborn 画：按性别的生还率 和 人数分布
import seaborn as sns

fig,axs = plt.subplots(1,2, figsize=(11,5))
sns.barplot(data=df, y=dep, x="Sex", ax=axs[0]).set(title="Survival rate")
sns.countplot(data=df, x="Sex", ax=axs[1]).set(title="Histogram");

In [ ]:
# 随机划分训练/验证集，并把类别列转成整数编码
from numpy import random
from sklearn.model_selection import train_test_split

random.seed(42)
trn_df,val_df = train_test_split(df, test_size=0.25)
trn_df[cats] = trn_df[cats].apply(lambda x: x.cat.codes)
val_df[cats] = val_df[cats].apply(lambda x: x.cat.codes)

In [ ]:
# 定义 xs_y：拆出自变量 xs 和因变量 y
def xs_y(df):
    xs = df[cats+conts].copy()
    return xs,df[dep] if dep in df else None

trn_xs,trn_y = xs_y(trn_df)
val_xs,val_y = xs_y(val_df)

In [ ]:
# 最简基线：预测“女性(Sex==0) 即生还”
preds = val_xs.Sex==0

In [ ]:
# 用平均绝对误差(MAE)评估这个基线
from sklearn.metrics import mean_absolute_error
mean_absolute_error(val_y, preds)

In [ ]:
# 画 LogFare 与生还的关系（boxen + 核密度）
df_fare = trn_df[trn_df.LogFare>0]
fig,axs = plt.subplots(1,2, figsize=(11,5))
sns.boxenplot(data=df_fare, x=dep, y="LogFare", ax=axs[0])
sns.kdeplot(data=df_fare, x="LogFare", ax=axs[1]);

In [ ]:
# 换个基线：LogFare>2.7 即生还
preds = val_xs.LogFare>2.7

In [ ]:
# 评估这个基线
mean_absolute_error(val_y, preds)

In [ ]:
# 一侧样本的“不纯度”打分：标准差 × 样本数（越小越好）
def _side_score(side, y):
    tot = side.sum()
    if tot<=1: return 0
    return y[side].std()*tot

In [ ]:
# 一个二分切分的总得分 = 左右两侧得分之和 / 总样本数
def score(col, y, split):
    lhs = col<=split
    return (_side_score(lhs,y) + _side_score(~lhs,y))/len(y)

In [ ]:
# 用 Sex 在 0.5 处切分的得分
score(trn_xs["Sex"], trn_y, 0.5)

In [ ]:
# 用 LogFare 在 2.7 处切分的得分
score(trn_xs["LogFare"], trn_y, 2.7)

In [ ]:
# 交互式：对连续列，滑动切分点看得分
def iscore(nm, split):
    col = trn_xs[nm]
    return score(col, trn_y, split)

from ipywidgets import interact
interact(nm=conts, split=15.5)(iscore);

In [ ]:
# 交互式：对类别列看得分
interact(nm=cats, split=2)(iscore);

In [ ]:
# 取 Age 的所有唯一值（排序），准备逐个当切分点试
nm = "Age"
col = trn_xs[nm]
unq = col.unique()
unq.sort()
unq

In [ ]:
# 找出让 Age 得分最低的切分点
scores = np.array([score(col, trn_y, o) for o in unq if not np.isnan(o)])
unq[scores.argmin()]

In [ ]:
# min_col：对某列找最优切分点及其得分
def min_col(df, nm):
    col,y = df[nm],df[dep]
    unq = col.dropna().unique()
    scores = np.array([score(col, y, o) for o in unq if not np.isnan(o)])
    idx = scores.argmin()
    return unq[idx],scores[idx]

min_col(trn_df, "Age")

In [ ]:
# 对所有列各找一次最优切分（这就是决策树第一层怎么选特征的）
cols = cats+conts
{o:min_col(trn_df, o) for o in cols}

In [ ]:
# 按 Sex 切成 男/女 两组，准备各自再找最优切分（手动长第二层）
cols.remove("Sex")
ismale = trn_df.Sex==1
males,females = trn_df[ismale],trn_df[~ismale]

In [ ]:
# 男性组里各列的最优切分
{o:min_col(males, o) for o in cols}

In [ ]:
# 女性组里各列的最优切分
{o:min_col(females, o) for o in cols}

In [ ]:
# 用 sklearn 直接训练一棵决策树（最多 4 个叶子）
from sklearn.tree import DecisionTreeClassifier, export_graphviz

m = DecisionTreeClassifier(max_leaf_nodes=4).fit(trn_xs, trn_y);

In [ ]:
# 定义 draw_tree：把决策树画成 graphviz 图
import graphviz

def draw_tree(t, df, size=10, ratio=0.6, precision=2, **kwargs):
    s=export_graphviz(t, out_file=None, feature_names=df.columns, filled=True, rounded=True,
                      special_characters=True, rotate=False, precision=precision, **kwargs)
    return graphviz.Source(re.sub('Tree {', f'Tree {{ size={size}; ratio={ratio}', s))

In [ ]:
# 画出这棵树
draw_tree(m, trn_xs, size=10)

In [ ]:
# gini 不纯度函数
def gini(cond):
    act = df.loc[cond, dep]
    return 1 - act.mean()**2 - (1-act).mean()**2

In [ ]:
# 看按性别分的 gini
gini(df.Sex=='female'), gini(df.Sex=='male')

In [ ]:
# 决策树在验证集上的 MAE
mean_absolute_error(val_y, m.predict(val_xs))

In [ ]:
# 训练更深的树（每个叶子至少 50 样本）并画出来
m = DecisionTreeClassifier(min_samples_leaf=50)
m.fit(trn_xs, trn_y)
draw_tree(m, trn_xs, size=25)

In [ ]:
# 这棵树的验证集 MAE
mean_absolute_error(val_y, m.predict(val_xs))

In [ ]:
# 对测试集预测并生成提交文件 sub-tree.csv
tst_df[cats] = tst_df[cats].apply(lambda x: x.cat.codes)
tst_xs,_ = xs_y(tst_df)

def subm(preds, suff):
    tst_df['Survived'] = preds
    sub_df = tst_df[['PassengerId','Survived']]
    sub_df.to_csv(f'sub-{suff}.csv', index=False)

subm(m.predict(tst_xs), 'tree')

In [ ]:
# 看 Embarked 原始值
df.Embarked.head()

In [ ]:
# 看 Embarked 的分类编码
df.Embarked.cat.codes.head()

In [ ]:
# get_tree：对数据随机采样一部分训练一棵树（bagging 的基础）
def get_tree(prop=0.75):
    n = len(trn_y)
    idxs = random.choice(n, int(n*prop))
    return DecisionTreeClassifier(min_samples_leaf=5).fit(trn_xs.iloc[idxs], trn_y.iloc[idxs])

In [ ]:
# 建 100 棵这样的随机树
trees = [get_tree() for t in range(100)]

In [ ]:
# 100 棵树预测取平均（手写随机森林）→ MAE
all_probs = [t.predict(val_xs) for t in trees]
avg_probs = np.stack(all_probs).mean(0)

mean_absolute_error(val_y, avg_probs)

In [ ]:
# 用 sklearn 的 RandomForestClassifier（100 棵树）
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(100, min_samples_leaf=5)
rf.fit(trn_xs, trn_y);
mean_absolute_error(val_y, rf.predict(val_xs))

In [ ]:
# 随机森林对测试集预测并生成提交文件
subm(rf.predict(tst_xs), 'rf')

In [ ]:
# 画特征重要性条形图（注意：这里用的是上面那棵单树 m 的 feature_importances_）
pd.DataFrame(dict(cols=trn_xs.columns, imp=m.feature_importances_)).plot('cols', 'imp', 'barh');